# NB6 · A web callable interface

**Building Clinical Decision Support Systems with Generative AI Tools**  
Technology and Artificial Intelligence Literacy Training in Health Sciences · Akdeniz University · 18 September 2026

Prof. Dr. Utku Köse · Süleyman Demirel University, Department of Computer Engineering  
Director, Artificial Intelligence Application and Research Center (YAZEM) · utkukose@sdu.edu.tr

---

In this final notebook the system built in the previous steps is connected to an
interface usable from a browser. Gradio produces a temporary public link inside Colab.

There is a check cell only at the end. By this stage you are able to read what the code
you generated does.


## Setup


In [ ]:
!pip -q install pandas numpy scikit-learn matplotlib

import urllib.request

REPO = 'https://raw.githubusercontent.com/utkukose/cdss-genai-NB-lecture/main'
for module in ['checks.py', 'evaluate.py', 'explain.py', 'safety.py',
               'mimic_web.py', 'pipeline.py']:
    urllib.request.urlretrieve(f'{REPO}/workshop/{module}', module)

import numpy as np
import pandas as pd
import checks, evaluate as ev, explain as ex, safety as sf

checks.LANG = ev.LANG = sf.LANG = 'en'

!pip -q install gradio


In [ ]:
# Fixed cell. Rebuilds the output of the previous notebooks.
import pipeline as pl

state = pl.prepare(verbose=False)
model = state['model']
test = state['test']
features = state['features']
probability = state['probabilities']
y_test = state['y_test']
threshold = ev.threshold_for_sensitivity(y_test, probability, target=0.80)
print(f'Model and predictions ready. Operating threshold: {threshold:.3f}')


In [ ]:
# Fixed cell. Assembles the guarded system.
band = sf.choose_band(y_test, probability, threshold, max_abstain=0.20)
guarded = sf.GuardedModel(
    model,
    sf.AbstentionPolicy(threshold, band['band']),
    sf.DriftDetector().fit(state['train'][features]),
    sf.PhysiologicalValidator().fit(state['train'][features]),
)

importance = ex.permutation_global(model, test[features], y_test, n_repeats=10, top=40)
numeric = set(state['train'][features].select_dtypes(include='number').columns)
SLIDERS = [f for f in importance['feature'] if f in numeric][:6]
CATEGORICAL = [f for f in features if f not in numeric][:2]
DEFAULTS = state['train'][features].median(numeric_only=True)

print('Sliders:', SLIDERS)
print('Dropdowns:', CATEGORICAL)


---

## What the interface shows

Placing a slider for more than thirty features makes the interface unusable. The six
most important numeric features are selected and the rest are held at the training
median. The interface must state that the features the user cannot see are held at default
values.

The screen carries the decision itself, the state of the guardrails and the contribution
breakdown rather than a single probability. This was the distinction drawn in the
lecture: the product of a decision support system is not a diagnosis but an allocation
of attention, presented together with its reasoning.

One further point requires care. Demographic attributes cannot be presented as clinical
reasons. Where they appear in the contribution table they must be moved to a separate
warning line. Such a contribution is a fairness audit finding, not a reason to show a
clinician.


### Prompt 1

```
Write a Python cell that builds a single page interface with Gradio.

I have the following:
  guarded     -> its predict_one method takes a single row DataFrame and returns a
                 dictionary containing the keys decision and probability
  SLIDERS     -> list of numeric column names to give sliders
  CATEGORICAL -> list of categorical column names to give dropdowns
  DEFAULTS    -> median values of the numeric columns
  features    -> every column name the model uses
  state['train'] -> the training set DataFrame

The interface should be laid out as follows:
1. A title at the top with this warning: This is a teaching prototype, not a
   validated clinical tool; it cannot be used for real patient decisions.
2. On the left, a slider for each SLIDERS column, a dropdown for each CATEGORICAL
   column, and an Evaluate button.
3. On the right, the decision as text and the contribution table.
4. State on screen how many features are not shown and that they are held at the
   median.
5. If a row in the contribution table mentions gender, race, insurance or
   marital_status, remove it from the table and show a separate fairness audit
   warning instead.

Take slider bounds from the 1st and 99th percentiles of the training set.

CONTRACT
Produce a function named evaluate that takes the slider and dropdown values in order
and returns two values: markdown text and a DataFrame.
Produce a Gradio Blocks object named interface.
Call interface.launch(share=True) on the final line.
```


In [ ]:
# Paste the generated code into this cell and run it.


---

## End of notebook · Interface check

The cell below is fixed. It calls the function directly, without opening the interface,
and shows what it returns in two situations.


In [ ]:
inputs = ([float(DEFAULTS.get(s, 0)) for s in SLIDERS] +
          [state['train'][c].mode().iloc[0] for c in CATEGORICAL])

text, table = evaluate(*inputs)
print('ORDINARY PATIENT'); print(text); print()

extreme = [float(state['train'][s].quantile(0.99)) * 40 for s in SLIDERS]
text_extreme, _ = evaluate(*(extreme + inputs[len(SLIDERS):]))
print('PATIENT WITH EXTREME VALUES'); print(text_extreme.split(chr(10))[0])


## What to try in the interface

Make five attempts.

Run it at the default values and observe the decision. Then move the sliders slowly to
bring the probability towards the threshold and find the point at which the system
begins to abstain. In clinical use that band determines who goes on the list and who is
left to the clinician.

Pull several sliders to their extremes. The system should stop producing predictions and
return an escalation message.

Reach a similar probability through different slider combinations and watch the
contribution table change. Two patients may carry the same risk while requiring
different action from the clinician. This is where showing only a number on screen
proves inadequate.

Finally change the sex and touch nothing else. If the probability moves and a fairness
audit warning appears, the model has learned from a demographic attribute. Such a
finding can be sufficient to prevent deployment.


## End of the workshop

The prototype works, and that does not mean the system is ready. Only the first links of
the chain described in the lecture have been completed.

What remains: External validation at another centre, physiological limits supplied by a
clinician, a workflow integration study, approval of the threshold by the clinical team,
regulatory classification, a data protection assessment and post-deployment performance
monitoring.

That list shows where two hours of work sits on the path to a clinical product. The cost
of producing code has fallen; nothing else on the list has.
---

**Warning.** Nothing produced in this notebook is a validated clinical tool. The
MIMIC-IV demo data comes from a single hospital in the United States and does not
represent an intensive care population elsewhere. The material is for teaching.
